In [ ]:

install.packages(c("MASS", "ordinal", "brant", "matrixcalc", "margins"))

library(MASS)
library(ordinal)
library(brant)
library(matrixcalc)
library(margins)

# 2. 数据加载 & 预处理

df <- read.csv("/content/crash_data (2).csv")

cat("\n原始 grav 变量分布:\n")
print(table(df$grav))

# 重编码,将fatal和hospitalized合并为severehosp类
df$grav_new <- NA
df$grav_new[df$grav %in% c(0, 3)] <- "SevereHosp"
df$grav_new[df$grav == 1] <- "Slight"
df$grav_new[df$grav == 2] <- "NoInjury"

# 转换成有序因子
df$grav_new <- factor(df$grav_new,
                      levels = c("SevereHosp", "Slight", "NoInjury"),
                      ordered = TRUE)

cat("\n重编码后因变量分布:\n")
print(table(df$grav_new))

df <- subset(df, secu %in% c(0,1))


df$secu <- factor(df$secu, levels = c(0,1), labels = c("NoBelt","Seatbelt"))

# 4. 分类变量转为因子（dummy 自动生成）

df$sexe <- factor(df$sexe)
df$obs  <- factor(df$obs)
df$obsm <- factor(df$obsm)
df$lum  <- factor(df$lum)
df$plan <- factor(df$plan)
df$prof <- factor(df$prof)
df$atm  <- factor(df$atm)

# 设置参考水平
df$lum  <- relevel(df$lum, ref = "1")
# Lighting conditions: 1=Daylight, 2=Dawn/Dusk, 3=Night without streetlights, 4=Night without streetlights, 5=Night with streetlights
df$plan <- relevel(df$plan, ref = "1")#1:straight; 0:not straight
df$prof <- relevel(df$prof, ref = "1")#1:flat;0:not flat
df$atm  <- relevel(df$atm, ref = "1")#0:bad weather;1:normal weather
df$sexe <- relevel(df$sexe, ref = "1")#1:male;2:female
df$obs  <- relevel(df$obs, ref = "0")#no fixed obstacle
df$obsm <- relevel(df$obsm, ref = "0")#no moving obstacle

# 5. 基线 Ologit 模型
base_model <- polr(grav_new ~ age * secu + nbv + sexe + obs + obsm + lum + plan + prof + atm,
                   data = df, Hess = TRUE, method = "logistic")
summary(base_model)


# 6. Brant Test (平行线假设检验)

cat("\n=== Brant Test (平行线假设检验) ===\n")
brant_result <- brant(base_model)
print(brant_result)


#  7. PPO 模型 (部分比例优势: age, nbv, sexe, lum 非比例)

fit_ppo <- clm(grav_new ~ age * secu + obs + obsm + plan + prof + atm + secu,
               nominal = ~ age + nbv + sexe + lum,   # 只放宽这4个变量
               data = df, link = "logit")
summary(fit_ppo)



# 8. 收敛性诊断

cat("\n=== 收敛性诊断 ===\n")
optinfo <- fit_ppo$optinfo
cat("迭代次数:", optinfo$niter, "\n")
cat("收敛标记 (0=收敛):", optinfo$conv, "\n")
if (!is.null(optinfo$grad)) {
  cat("最大梯度:", max(abs(optinfo$grad)), "\n")
} else {
  cat("最大梯度: NA (模型完全收敛)\n")
}
V <- vcov(fit_ppo)
cat("Hessian 正定性:", is.positive.definite(as.matrix(V)), "\n")

# 9. LR Test (与平行线模型比较)

cat("\n=== LR Test ===\n")
model_parallel <- clm(grav_new ~ age * secu + age + nbv + sexe + obs + obsm + lum + plan + prof + atm + secu,
                      data = df, link = "logit")
print(anova(model_parallel, fit_ppo))

# 10. 稳健性检验（三次拟合对比）

cat("\n=== 稳健性检验: 三次拟合对比 ===\n")
fits <- list()
for (i in 1:3) {
  fits[[i]] <- clm(grav_new ~ age * secu + obs + obsm + plan + prof + atm + secu,
                   nominal = ~ age + nbv + sexe + lum,
                   data = df, link="logit")
}
coef_compare <- sapply(fits, coef)
print(round(coef_compare, 4))


# # 11. 边际效应 (近似 Ordered Logit)
# cat("\n=== 边际效应 (近似) ===\n")
# fit_ologit <- polr(grav_new ~ age * secu + nbv + sexe + obs + obsm + lum + plan + prof + atm,
#                    data = df, Hess = TRUE, method = "logistic")
# mfx <- margins(fit_ologit)
# print(summary(mfx))
#marginal effect cannot be calcualeted based on R, as it;s not correct!


# 12. 保存 PPO 模型
saveRDS(fit_ppo, "ppo_model.rds")

# 13. 模型比较
cat("\n=== 模型比较 ===\n")
models_comparison <- data.frame(
  Model = c("Ordered Logit", "PPO"),
  AIC = c(AIC(model_parallel), AIC(fit_ppo)),
  BIC = c(BIC(model_parallel), BIC(fit_ppo)),
  LogLik = c(as.numeric(logLik(model_parallel)), as.numeric(logLik(fit_ppo)))
)
print(models_comparison)

cat("\n=== 额外诊断指标 ===\n")
cat("AIC:", AIC(fit_ppo), "\n")
cat("BIC:", BIC(fit_ppo), "\n")
cat("Log-Likelihood:", logLik(fit_ppo), "\n")


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘ucminf’, ‘numDeriv’, ‘prediction’





原始 grav 变量分布:

   0    1    2    3 
   6 2977 3981  493 

重编码后因变量分布:

SevereHosp     Slight   NoInjury 
       499       2977       3981 


Call:
polr(formula = grav_new ~ age * secu + nbv + sexe + obs + obsm + 
    lum + plan + prof + atm, data = df, Hess = TRUE, method = "logistic")

Coefficients:
                    Value Std. Error  t value
age               0.02140   0.009298   2.3015
secuSeatbelt      2.13500   0.351428   6.0752
nbv              -0.02752   0.011986  -2.2961
sexe2            -0.63004   0.048199 -13.0715
obs1             -0.98845   0.103036  -9.5933
obsm1             0.22277   0.113190   1.9681
lum2             -0.02085   0.110711  -0.1883
lum3              0.10611   0.211585   0.5015
lum4              0.13254   0.271457   0.4883
lum5             -0.25509   0.054098  -4.7153
plan0            -0.11741   0.073504  -1.5974
prof0            -0.04158   0.054762  -0.7592
atm0             -0.17826   0.076388  -2.3337
age:secuSeatbelt -0.01180   0.009398  -1.2560

Intercepts:
                  Value    Std. Error t value 
SevereHosp|Slight  -0.5845   0.3626    -1.6117
Slight|NoInjury     2.0713   0.3648     5.


=== Brant Test (平行线假设检验) ===
---------------------------------------------------- 
Test for		X2	df	probability 
---------------------------------------------------- 
Omnibus			63.58	14	0
age			5.27	1	0.02
secuSeatbelt		2.11	1	0.15
nbv			8.53	1	0
sexe2			12.43	1	0
obs1			3.87	1	0.05
obsm1			0	1	1
lum2			1.93	1	0.16
lum3			17.45	1	0
lum4			0.06	1	0.8
lum5			0.06	1	0.81
plan0			0.45	1	0.5
prof0			0.3	1	0.59
atm0			4.26	1	0.04
age:secuSeatbelt	2.76	1	0.1
---------------------------------------------------- 

H0: Parallel Regression Assumption holds


Warning message in brant(base_model):
“1536 combinations in table(dv,ivs) do not occur. Because of that, the test results might be invalid.”


                           X2 df  probability
Omnibus          6.357715e+01 14 2.741296e-08
age              5.270425e+00  1 2.169072e-02
secuSeatbelt     2.108274e+00  1 1.465045e-01
nbv              8.527668e+00  1 3.497875e-03
sexe2            1.242654e+01  1 4.232752e-04
obs1             3.870066e+00  1 4.915458e-02
obsm1            9.291842e-07  1 9.992309e-01
lum2             1.934784e+00  1 1.642360e-01
lum3             1.744917e+01  1 2.950931e-05
lum4             6.366511e-02  1 8.007942e-01
lum5             5.509595e-02  1 8.144220e-01
plan0            4.535262e-01  1 5.006651e-01
prof0            2.960956e-01  1 5.863405e-01
atm0             4.257111e+00  1 3.908634e-02
age:secuSeatbelt 2.756164e+00  1 9.688030e-02


formula: grav_new ~ age * secu + obs + obsm + plan + prof + atm + secu
nominal: ~age + nbv + sexe + lum
data:    df

 link  threshold nobs logLik   AIC      niter max.grad cond.H 
 logit flexible  7373 -6181.50 12409.01 6(0)  2.47e-09 2.8e+06

Coefficients: (1 not defined because of singularities)
                 Estimate Std. Error z value Pr(>|z|)    
age                    NA         NA      NA       NA    
secuSeatbelt      2.23960    0.35786   6.258 3.89e-10 ***
obs1             -0.99212    0.10332  -9.602  < 2e-16 ***
obsm1             0.22154    0.11353   1.951   0.0510 .  
plan0            -0.11246    0.07370  -1.526   0.1271    
prof0            -0.04291    0.05497  -0.781   0.4351    
atm0             -0.18343    0.07666  -2.393   0.0167 *  
age:secuSeatbelt -0.01480    0.00939  -1.576   0.1150    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Threshold coefficients:
                               Estimate Std. Error z value
SevereHosp|Slight.(Intercept


=== 收敛性诊断 ===
迭代次数: 
收敛标记 (0=收敛): 
最大梯度: NA (模型完全收敛)
Hessian 正定性: TRUE 

=== LR Test ===
Likelihood ratio tests of cumulative link models:
 
               formula:                                                                              
model_parallel grav_new ~ age * secu + age + nbv + sexe + obs + obsm + lum + plan + prof + atm + secu
fit_ppo        grav_new ~ age * secu + obs + obsm + plan + prof + atm + secu                         
               nominal:                link: threshold:
model_parallel ~1                      logit flexible  
fit_ppo        ~age + nbv + sexe + lum logit flexible  

               no.par   AIC  logLik LR.stat df Pr(>Chisq)    
model_parallel     16 12436 -6201.8                          
fit_ppo            23 12409 -6181.5  40.599  7  9.666e-07 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

=== 稳健性检验: 三次拟合对比 ===
                                 [,1]    [,2]    [,3]
SevereHosp|Slight.(Intercept) -0.8346 -0.8346 -0.8346

In [ ]:
#主文使用 parametric bootstrap（基于系数分布） → 快、标准、主流。

import numpy as np
import pandas as pd
from scipy.special import expit  # logistic CDF
from tqdm import tqdm

# ===================================================
# 1. 从 R 输出抄过来的 PPO 结果
# ===================================================
coefs_point = {
    "age": 0.0099,
    "nbv": -0.0275,
    "sexe2": -0.6289,
    "obs1": -0.987,
    "obsm1": 0.224,
    "lum2": -0.023,
    "lum3": 0.106,
    "lum4": 0.136,
    "lum5": -0.256,
    "plan0": -0.113,
    "prof0": -0.045,
    "atm0": -0.179
}

# 系数标准误（来自 R 输出）
coefs_se = {
    "age": 0.001426,
    "nbv": 0.011985,
    "sexe2": 0.048196,
    "obs1": 0.103014,
    "obsm1": 0.113208,
    "lum2": 0.110678,
    "lum3": 0.211618,
    "lum4": 0.271642,
    "lum5": 0.054099,
    "plan0": 0.073451,
    "prof0": 0.054759,
    "atm0": 0.076389
}

coef_names = list(coefs_point.keys())

# 阈值（含 secu 非比例效应）
thresholds = {
    "SevereHosp|Slight.Intercept": -0.9489,
    "Slight|NoInjury.Intercept": 1.6736,
    "SevereHosp|Slight.secuSeatbelt": -1.7571,
    "Slight|NoInjury.secuSeatbelt": -1.7242
}

# ===================================================
# 2. PPO 预测函数
# ===================================================
def ppo_predict(x, secu_value, coefs, thresholds):
    xb = sum([x.get(k, 0) * v for k, v in coefs.items()])
    tau1 = thresholds["SevereHosp|Slight.Intercept"] + secu_value * thresholds["SevereHosp|Slight.secuSeatbelt"]
    tau2 = thresholds["Slight|NoInjury.Intercept"] + secu_value * thresholds["Slight|NoInjury.secuSeatbelt"]

    p1 = expit(tau1 - xb)                 # SevereHosp
    p2 = expit(tau2 - xb) - expit(tau1 - xb)  # Slight
    p3 = 1 - expit(tau2 - xb)             # NoInjury
    return np.array([p1, p2, p3])

# ===================================================
# 3. 边际效应计算
# ===================================================
def marginal_effect(x, var, coefs, thresholds, var_type="numeric", h=1e-5):
    if var_type == "numeric":
        x_low, x_high = x.copy(), x.copy()
        x_low[var] -= h
        x_high[var] += h
        p_low = ppo_predict(x_low, 1, coefs, thresholds)
        p_high = ppo_predict(x_high, 1, coefs, thresholds)
        return (p_high - p_low) / (2*h)
    elif var_type == "categorical":
        x0, x1 = x.copy(), x.copy()
        x0[var] = 0
        x1[var] = 1
        p0 = ppo_predict(x0, 1, coefs, thresholds)
        p1 = ppo_predict(x1, 1, coefs, thresholds)
        return p1 - p0

# ===================================================
# 4. Parametric Bootstrap CI
# ===================================================
def bootstrap_ci_parametric(x, var, coef_means, coef_ses, thresholds, var_type="numeric", R=500):
    effects = []
    for _ in range(R):
        # 每次抽一套新的系数
        sampled_coefs = {k: np.random.normal(coef_means[k], coef_ses[k]) for k in coef_means}
        eff = marginal_effect(x, var, sampled_coefs, thresholds, var_type)
        effects.append(eff)
    effects = np.array(effects)
    ame = effects.mean(axis=0)
    ci_lower = np.percentile(effects, 2.5, axis=0)
    ci_upper = np.percentile(effects, 97.5, axis=0)
    return ame, ci_lower, ci_upper

# ===================================================
# 5. 示例：计算所有变量的边际效应 + CI
# ===================================================
# 设定一个基准个体（基类水平都设为 0）
X_example = {
    "age": 40,
    "nbv": 2,
    "sexe2": 0,   # female baseline
    "obs1": 0,
    "obsm1": 0,
    "lum2": 0, "lum3": 0, "lum4": 0, "lum5": 0,  # lum1 baseline
    "plan0": 0,
    "prof0": 0,
    "atm0": 0
}

var_types = {
    "age": "numeric",
    "nbv": "numeric",
    "sexe2": "categorical",
    "obs1": "categorical",
    "obsm1": "categorical",
    "lum2": "categorical",
    "lum3": "categorical",
    "lum4": "categorical",
    "lum5": "categorical",
    "plan0": "categorical",
    "prof0": "categorical",
    "atm0": "categorical"
}

results = []
for v, t in tqdm(var_types.items()):
    ame, ci_l, ci_u = bootstrap_ci_parametric(
        X_example, v, coefs_point, coefs_se, thresholds, var_type=t, R=500
    )
    for k, outcome in enumerate(["SevereHosp", "Slight", "NoInjury"]):
        results.append({
            "Variable": v,
            "Outcome": outcome,
            "AME": ame[k],
            "CI_lower": ci_l[k],
            "CI_upper": ci_u[k]
        })

df_results = pd.DataFrame(results)
print(df_results)

# 保存结果
df_results.to_csv("ppo_marginal_effects_bootstrap.csv", index=False)
print("\n✅ 已保存为 ppo_marginal_effects_bootstrap.csv")


ERROR: Error in parse(text = input): <text>:3:8: unexpected symbol
2: 
3: import numpy
          ^
